# CapCap GPU Server - Chuyên dụng cho Bóc băng (Whisper)
Sổ tay này tạo một server FastAPI độc lập và tối giản trên Google Colab chỉ cài đặt các thư viện cần thiết cho việc bóc băng (Faster-Whisper), không cần clone toàn bộ mã nguồn của ứng dụng.

**Hướng dẫn:**
1. Vào menu **Runtime** > **Change runtime type** > Chọn **T4 GPU**.
2. Bấm nút **Play** ở ô bên dưới để chạy mã.
3. Đợi hệ thống cài đặt và cấp cho bạn 1 URL (đuôi `.trycloudflare.com`) cùng 1 Token bảo mật.
4. Mở file `.env` của CapCap trên máy tính cục bộ của bạn, dán URL và Token đó vào `CAPCAP_REMOTE_API_URL` và `CAPCAP_REMOTE_API_TOKEN`.

In [ ]:
import os
import subprocess
import time
import secrets
import IPython.display as display

# 1. Cài đặt thư viện cần thiết
print("Đang cài đặt faster-whisper và fastapi (có thể mất vài phút)...")
!pip install faster-whisper fastapi uvicorn pydub python-multipart > /dev/null 2>&1

# 2. Cài đặt Cloudflare Tunnel
!curl -s -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1

# 3. Tạo file script chạy FastAPI Server độc lập cho Whisper
server_code = """
import os
import base64
import tempfile
from fastapi import FastAPI, Request, HTTPException
from pydantic import BaseModel
from faster_whisper import WhisperModel

app = FastAPI()

# Biến global lưu model để không phải tải lại mỗi lần request
loaded_model = None
loaded_model_name = ""

class TranscribeRequest(BaseModel):
    audio_b64: str
    audio_filename: str = "remote_input.wav"
    model_name: str = "base"
    language: str = "auto"
    task: str = "transcribe"

@app.get("/health")
def health():
    return {"ok": True, "service": "capcap-remote-whisper-colab"}

@app.post("/v1/transcribe")
def transcribe(req: TranscribeRequest, request: Request):
    global loaded_model, loaded_model_name
    
    # Kiểm tra Token bảo mật
    expected_token = os.environ.get("CAPCAP_REMOTE_API_TOKEN")
    if expected_token:
        supplied_token = request.headers.get("X-CapCap-Token", "")
        if supplied_token != expected_token:
            raise HTTPException(status_code=401, detail="Invalid remote API token.")

    # Khởi tạo model nếu chưa có hoặc đổi model
    if loaded_model is None or loaded_model_name != req.model_name:
        print(f"Loading Whisper model: {req.model_name}")
        loaded_model = WhisperModel(req.model_name, device="cuda", compute_type="float16")
        loaded_model_name = req.model_name

    # Giải mã file âm thanh base64
    audio_bytes = base64.b64decode(req.audio_b64)
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(audio_bytes)
        tmp_path = f.name

    try:
        # Nhận diện âm thanh
        lang = req.language if req.language != "auto" else None
        segments, info = loaded_model.transcribe(tmp_path, language=lang, task=req.task, word_timestamps=True)
        
        result_segments = []
        for segment in segments:
            words = []
            if segment.words:
                for w in segment.words:
                    words.append({
                        "start": float(w.start),
                        "end": float(w.end),
                        "text": str(w.word).strip()
                    })
            
            result_segments.append({
                "start": float(segment.start),
                "end": float(segment.end),
                "text": segment.text.strip(),
                "words": words
            })
            
        return {"ok": True, "segments": result_segments}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        os.remove(tmp_path)
"""

with open("whisper_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

# 4. Thiết lập môi trường và Token bảo mật
TOKEN = secrets.token_urlsafe(32)
os.environ["CAPCAP_REMOTE_API_TOKEN"] = TOKEN

# 5. Khởi chạy Server
print("Đang khởi động CapCap Whisper Server...")
server_process = subprocess.Popen(
    ["uvicorn", "whisper_server:app", "--host", "127.0.0.1", "--port", "8765"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy()
)
time.sleep(5)

# 6. Khởi chạy Cloudflare Tunnel
print("Đang thiết lập Cloudflare Tunnel...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 7. Lấy URL Public từ Cloudflare
public_url = ""
while True:
    line = tunnel.stdout.readline()
    if not line:
        break
    if "https://" in line and ".trycloudflare.com" in line:
        import re
        match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

display.clear_output()
print("\n" + "="*70)
print("✅ MÁY CHỦ COLAB WHISPER ĐÃ SẴN SÀNG ✅")
print("Hãy copy 2 dòng sau và dán vào file .env trên máy tính của bạn:")
print("="*70)
print(f"CAPCAP_REMOTE_API_URL={public_url}")
print(f"CAPCAP_REMOTE_API_TOKEN={TOKEN}")
print("="*70)
print("\nBạn có thể để tab này chạy nền. Đừng đóng trình duyệt nhé!\n")

# Hiển thị log của server liên tục
try:
    for line in iter(server_process.stdout.readline, ""):
        print(line, end="")
except KeyboardInterrupt:
    print("\nĐã dừng server.")
